# MachineGuard AI — Industrial Predictive Maintenance Research Notebook
## Academic Mini-Project: AI-Based Predictive Maintenance Using Machine Learning

**Lead Author / Engineer:** MachineGuard AI Team  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI Machine Learning Repository ID: 601)  
**Target:** Machine Failure (Binary Classification)

> **Data Honesty Statement:**  
> This notebook documents an educational prototype trained on the AI4I 2020 synthetic predictive-maintenance benchmark. Results are not validated for deployment on real industrial machinery. All failure mechanisms are modeled synthetically.


---
## 1. Environment Setup & Core Dependencies
We import standard data science and machine learning libraries: `pandas`, `numpy`, `scikit-learn`, `joblib`, and visualization libraries.


In [1]:
import math
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)

np.random.seed(42)
print("Libraries loaded successfully.")


Libraries loaded successfully.


---
## 2. Dataset Ingestion & Exploratory Data Analysis (EDA)
The canonical dataset is sourced from the UCI Machine Learning Repository (AI4I 2020 dataset). It consists of 10,000 synthetic records reflecting industrial CNC milling/cutting machine telemetry.


In [2]:
DATA_PATH = (
    os.path.join("..", "backend", "data", "raw", "ai4i2020.csv")
    if os.path.exists(os.path.join("..", "backend", "data", "raw", "ai4i2020.csv"))
    else os.path.join("backend", "data", "raw", "ai4i2020.csv")
)
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing Values across all columns:\n{df.isnull().sum()}")
df.head()

Dataset Shape: 10000 rows, 14 columns
Missing Values across all columns:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


### Analysis: Dataset Schema & Data Quality
* **Records:** 10,000 rows, 14 columns.
* **Missing values:** 0 missing values across all columns.
* **Type distribution:** L = 60.00% (6,000), M = 29.97% (2,997), H = 10.03% (1,003).
* **Numerical telemetry:** Air temperature [K], Process temperature [K], Rotational speed [rpm], Torque [Nm], and Tool wear [min].

In [3]:
failures = df["Machine failure"].value_counts()
failure_pct = df["Machine failure"].value_counts(normalize=True) * 100

print("Target Class Breakdown:")
print(f"  - Non-failures (0): {failures[0]} ({failure_pct[0]:.2f}%)")
print(f"  - Failures (1):     {failures[1]} ({failure_pct[1]:.2f}%)")


Target Class Breakdown:
  - Non-failures (0): 9661 (96.61%)
  - Failures (1):     339 (3.39%)


### Analysis: Severe Class Imbalance
The target variable `Machine failure` exhibits severe class imbalance:
* **Normal Operation (Class 0):** 9,661 instances (96.61%)
* **Failure (Class 1):** 339 instances (3.39%)

**Implication for Modeling:**  
A naive classifier predicting "No Failure" on every sample achieves **96.61% accuracy** while offering **0% utility** in an industrial setting. Therefore:
1. Accuracy is disqualified as a primary selection metric.
2. We must prioritize **PR-AUC (Precision-Recall Area Under Curve)**, **Recall (Sensitivity)** on the minority class, and **F1-Score**.
3. We must apply `class_weight="balanced"` to penalize false negatives.


---
## 3. Target Leakage Prevention & Predictor Selection
The dataset includes several failure-mode indicator flags:
* `TWF` (Tool Wear Failure)
* `HDF` (Heat Dissipation Failure)
* `PWF` (Power Failure)
* `OSF` (Overstrain Failure)
* `RNF` (Random Failure)
* `UDI` (Row identifier)
* `Product ID` (Serial string)

> **CRITICAL RULE:** These columns describe specific post-failure root causes or arbitrary identifiers. Utilizing them as model inputs creates **catastrophic target leakage**. They are strictly removed from the feature set.


In [4]:
DROP_COLUMNS = ["UDI", "Product ID", "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
FEATURE_COLUMNS = [c for c in df.columns if c not in DROP_COLUMNS]

print(f"Usable Model Predictors ({len(FEATURE_COLUMNS)} features):")
for col in FEATURE_COLUMNS:
    print(f"  - {col}")


Usable Model Predictors (6 features):
  - Type
  - Air temperature [K]
  - Process temperature [K]
  - Rotational speed [rpm]
  - Torque [Nm]
  - Tool wear [min]


---
## 4. Physics-Informed Feature Engineering
Rather than hardcoding known synthetic threshold rules into the classifier, we derive three physical variables based on thermal dynamics and mechanics:

1. **`temperature_difference`**:
   $$\Delta T = T_{\text{process}} - T_{\text{air}}$$
   *Physical Meaning:* The thermal gradient reflects heat exchange efficiency. Inability to dissipate heat causes steep gradients or thermal stalls.

2. **`mechanical_power_kw`**:
   $$P_{\text{mech}} = \frac{\tau \cdot \omega}{1000} = \frac{\text{Torque [Nm]} \times \text{Rotational Speed [rpm]} \times 2\pi}{60 \times 1000}$$
   *Physical Meaning:* Mechanical output power transferred to the spindle drive. Extreme power excursions indicate drive overload or motor stall.

3. **`wear_load`**:
   $$\text{WearLoad} = \text{Tool wear [min]} \times \text{Torque [Nm]}$$
   *Physical Meaning:* Compound mechanical stress. High cutting resistance applied against a tool that has already experienced extensive micro-fractures causes sudden breakage.


In [5]:
def add_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    df_feat = data.copy()
    df_feat["temperature_difference"] = df_feat["Process temperature [K]"] - df_feat["Air temperature [K]"]
    df_feat["mechanical_power_kw"] = (
        df_feat["Torque [Nm]"] * df_feat["Rotational speed [rpm]"] * (2.0 * math.pi) / 60000.0
    )
    df_feat["wear_load"] = df_feat["Tool wear [min]"] * df_feat["Torque [Nm]"]
    return df_feat

df_engineered = add_engineered_features(df)
df_engineered[["temperature_difference", "mechanical_power_kw", "wear_load"]].describe()


,temperature_difference,mechanical_power_kw,wear_load
count,10000.000000,10000.000000,10000.000000
mean,10.000630,6.279745,4314.664550
std,1.001094,1.067418,2826.567692
min,7.600000,1.148441,0.000000
25%,9.300000,5.561184,1963.650000
50%,9.800000,6.271027,4012.950000
75%,11.000000,7.003003,6279.000000
max,12.100000,10.469923,16497.000000


---
## 5. Strict Featurization Ordering & Stratified Data Splitting
To adhere to rigorous ML best practices:
1. We split the data into **Train (70%)**, **Validation (15%)**, and **Test (15%)** sets.
2. We perform **stratification** on `Machine failure` to preserve the 3.39% failure ratio across all splits.
3. Preprocessing transformers (encoders and scalers) are **fit strictly on the Training set** to avoid data snooping.


In [6]:
y = df["Machine failure"].values
X = df[FEATURE_COLUMNS]

# Step 1: 70% Train, 30% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step 2: 15% Validation, 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train split:      {len(X_train)} samples ({y_train.sum()} failures, {y_train.mean():.2%})")
print(f"Validation split: {len(X_val)} samples ({y_val.sum()} failures, {y_val.mean():.2%})")
print(f"Test split:       {len(X_test)} samples ({y_test.sum()} failures, {y_test.mean():.2%})")


Train split:      7000 samples (237 failures, 3.39%)
Validation split: 1500 samples (51 failures, 3.40%)
Test split:       1500 samples (51 failures, 3.40%)


---
## 6. Ablation Comparison: Evaluating the Impact of Feature Engineering
We construct four pipeline configurations to isolate the effect of engineered features across both linear and tree-based paradigms:
1. **Baseline 1:** Logistic Regression (Original Usable Features)
2. **Baseline 2:** Logistic Regression (Original + Engineered Features)
3. **Model 1:** Random Forest Classifier (Original Usable Features)
4. **Model 2:** Random Forest Classifier (Original + Engineered Features)


In [7]:
CATEGORICAL_COLS = ["Type"]
NUMERIC_ORIGINAL = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"
]
ENGINEERED_COLS = ["temperature_difference", "mechanical_power_kw", "wear_load"]

def evaluate_candidate(model, include_engineered=True):
    num_cols = NUMERIC_ORIGINAL + (ENGINEERED_COLS if include_engineered else [])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(categories=[["L", "M", "H"]], drop="first", sparse_output=False), CATEGORICAL_COLS),
            ("num", StandardScaler(), num_cols)
        ]
    )
    
    X_tr = add_engineered_features(X_train) if include_engineered else X_train
    X_v = add_engineered_features(X_val) if include_engineered else X_val
    
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    
    pipe.fit(X_tr, y_train)
    val_probs = pipe.predict_proba(X_v)[:, 1]
    val_preds = (val_probs >= 0.5).astype(int)
    
    return {
        "F1-Score": f1_score(y_val, val_preds, zero_division=0),
        "PR-AUC": average_precision_score(y_val, val_probs),
        "ROC-AUC": roc_auc_score(y_val, val_probs),
        "Recall": recall_score(y_val, val_preds, zero_division=0),
        "Precision": precision_score(y_val, val_preds, zero_division=0),
        "pipeline": pipe,
        "val_probs": val_probs
    }

candidates = {
    "Logistic Regression (Original)": (LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000), False),
    "Logistic Regression (Engineered)": (LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000), True),
    "Random Forest (Original)": (RandomForestClassifier(n_estimators=150, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1), False),
    "Random Forest (Engineered)": (RandomForestClassifier(n_estimators=150, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1), True),
}

results = {}
for name, (clf, eng) in candidates.items():
    results[name] = evaluate_candidate(clf, include_engineered=eng)

summary_df = pd.DataFrame({
    k: {m: round(v[m], 4) for m in ["F1-Score", "PR-AUC", "ROC-AUC", "Recall", "Precision"]}
    for k, v in results.items()
}).T

summary_df


,F1-Score,PR-AUC,ROC-AUC,Recall,Precision
Logistic Regression (Original),0.2298,0.3834,0.8749,0.7255,0.1365
Logistic Regression (Engineered),0.2867,0.3562,0.9048,0.8039,0.1745
Random Forest (Original),0.6250,0.6639,0.9731,0.6863,0.5738
Random Forest (Engineered),0.8155,0.8479,0.9782,0.8235,0.8077


### Analysis: Key Findings from the Ablation Study
1. **Dramatic Gain from Feature Engineering:**
   * In the Random Forest model, adding engineered physical features drove the **Validation F1-score from 0.6250 to 0.8155 (+19.05% absolute increase)**.
   * **PR-AUC increased from 0.6639 to 0.8479 (+18.40%)**.
   * Precision surged from **0.5738 to 0.8077**, eliminating a vast number of false alarms while boosting recall to **82.35%**.
2. **Model Paradigm Comparison:**
   * Logistic Regression struggles with nonlinear boundary interactions between rotational speed and torque, topping out at F1 = 0.2867.
   * Random Forest with engineered features is decisively selected as the primary production architecture.


---
## 7. Validation-Guided Probability Threshold Tuning
In imbalanced classification, the default 0.50 decision threshold is rarely optimal. We tune the threshold **strictly on the Validation set** to optimize the F1-Score.


In [8]:
best_rf = results["Random Forest (Engineered)"]["pipeline"]
val_probs = results["Random Forest (Engineered)"]["val_probs"]

thresholds = np.linspace(0.1, 0.9, 81)
f1_scores = []
precisions = []
recalls = []

for t in thresholds:
    p = (val_probs >= t).astype(int)
    f1_scores.append(f1_score(y_val, p, zero_division=0))
    precisions.append(precision_score(y_val, p, zero_division=0))
    recalls.append(recall_score(y_val, p, zero_division=0))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Optimal Validation Threshold: {best_threshold:.2f}")
print(f"Validation F1 at Default 0.50: {results['Random Forest (Engineered)']['F1-Score']:.4f}")
print(f"Validation F1 at Tuned {best_threshold:.2f}:   {f1_scores[best_idx]:.4f} (Recall: {recalls[best_idx]:.4f}, Precision: {precisions[best_idx]:.4f})")


Optimal Validation Threshold: 0.56
Validation F1 at Default 0.50: 0.8155
Validation F1 at Tuned 0.56:   0.8367 (Recall: 0.8039, Precision: 0.8723)


---
## 8. Final Evaluation on the Untouched Test Set
Having locked the model architecture and tuned the decision threshold on validation data, we now evaluate the system **exactly once** on the untouched test partition.


In [9]:
X_test_eng = add_engineered_features(X_test)
test_probs = best_rf.predict_proba(X_test_eng)[:, 1]

# Default threshold 0.50
preds_default = (test_probs >= 0.50).astype(int)
cm_default = confusion_matrix(y_test, preds_default)

# Tuned threshold 0.56
preds_tuned = (test_probs >= best_threshold).astype(int)
cm_tuned = confusion_matrix(y_test, preds_tuned)

test_pr_auc = average_precision_score(y_test, test_probs)
test_roc_auc = roc_auc_score(y_test, test_probs)

print("=== FINAL TEST SET EVALUATION ===")
print(f"PR-AUC:  {test_pr_auc:.4f}")
print(f"ROC-AUC: {test_roc_auc:.4f}")
print(f"Default (0.50) -> F1: {f1_score(y_test, preds_default):.4f} | Recall: {recall_score(y_test, preds_default):.4f} | Precision: {precision_score(y_test, preds_default):.4f}")
print(f"Tuned ({best_threshold:.2f})   -> F1: {f1_score(y_test, preds_tuned):.4f} | Recall: {recall_score(y_test, preds_tuned):.4f} | Precision: {precision_score(y_test, preds_tuned):.4f} | Accuracy: {accuracy_score(y_test, preds_tuned):.4f}")

print("\nConfusion Matrix (Tuned Threshold):")
print(f"  TN: {cm_tuned[0,0]} | FP: {cm_tuned[0,1]}")
print(f"  FN: {cm_tuned[1,0]}  | TP: {cm_tuned[1,1]}")


=== FINAL TEST SET EVALUATION ===
PR-AUC:  0.8905
ROC-AUC: 0.9876
Default (0.50) -> F1: 0.7925 | Recall: 0.8235 | Precision: 0.7636
Tuned (0.56)   -> F1: 0.8235 | Recall: 0.8235 | Precision: 0.8235 | Accuracy: 0.9880

Confusion Matrix (Tuned Threshold):
  TN: 1440 | FP: 9
  FN: 9  | TP: 42


### Analysis: Test Set Generalization
* **ROC-AUC on Test:** **0.9876** (high rank-ordering capability across nominal and failure modes).
* **PR-AUC on Test:** **0.8905** (demonstrating strong discriminatory performance on the 3.39% rare failure class).
* **F1-Score at Tuned Threshold (0.56):** **0.8235** (optimal harmonic balance between precision and recall).
* **Recall at Tuned Threshold:** **82.35%** (correctly catching 42 out of 51 failures in the test set).
* **Precision at Tuned Threshold:** **82.35%** (only 9 false alarms out of 1,449 normal machines).
* **Accuracy at Tuned Threshold:** **98.80%** (1,482 out of 1,500 correct classifications).
* **Generalization Consistency:** The test results are consistent with the validation results, indicating that the selected pipeline retains strong performance on the held-out test partition.

---
## 9. Feature Importance & Explainability
We inspect the Gini importance values of the fitted Random Forest to understand what physical parameters drive the model's predictions.


In [10]:
rf_classifier = best_rf.named_steps["classifier"]
preprocessor = best_rf.named_steps["preprocessor"]

cat_cols = list(preprocessor.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_COLS))
num_cols = NUMERIC_ORIGINAL + ENGINEERED_COLS
all_feature_names = cat_cols + num_cols

feat_imp = pd.Series(rf_classifier.feature_importances_, index=all_feature_names).sort_values(ascending=False)

print("Random Forest Feature Importances:")
print(feat_imp.to_string())


Random Forest Feature Importances:
Rotational speed [rpm]     0.200960
mechanical_power_kw        0.194319
Torque [Nm]                0.181403
Tool wear [min]            0.141203
wear_load                  0.115903
temperature_difference     0.093815
Air temperature [K]        0.040503
Process temperature [K]    0.024526
Type_M                     0.005064
Type_H                     0.002304


### Analysis: Physical Validation of Engineered Features
* **`mechanical_power_kw`** emerged as the **#2 most influential feature (19.4% importance)**, just behind rotational speed (20.1%).
* **`wear_load`** ranks **#5 (11.6% importance)**, effectively capturing compound degradation before sudden failure occurs.
* **`temperature_difference`** contributes **9.4% importance**, providing the primary signal for heat dissipation failures.
* Combined, the three engineered features account for over **40.4% of total predictive importance**, confirming that domain feature engineering was essential to solving this task.


---
## 10. Summary & Conclusion
In this academic mini-project:
1. **Clean Baseline & Strict Separation:** We respected the synthetic AI4I benchmark by strictly omitting identifier and failure leakage columns (`UDI`, `Product ID`, `TWF`, `HDF`, `PWF`, `OSF`, `RNF`).
2. **Feature Engineering:** We derived three physically grounded features (`temperature_difference`, `mechanical_power_kw`, `wear_load`) which delivered an 18-19% absolute improvement in F1 and PR-AUC.
3. **Rigorous Validation:** Data was split into 70% Train, 15% Validation, and 15% Test. Model selection and threshold tuning (optimal threshold = 0.56) were conducted strictly on the validation set.
4. **Strong Test Results:** The resulting model achieved **0.8905 PR-AUC** and **0.8235 F1-score** on the untouched test partition.
5. **Production Readiness:** The pipeline is encapsulated into a self-contained scikit-learn artifact (`model.joblib`) ready for deployment in the FastAPI backend.
